In [0]:
#################################################
##############companies##########################
#################################################
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from delta.tables import DeltaTable

spark = SparkSession.builder.getOrCreate()
df = spark.read.table("investment_intelligence_platform.bronze.company")
df = df.select(
    col("company_number"),
    col("company_name"),
    col("date_of_creation"),
    col("has_been_liquidated"),
    col("has_charges"),
    col("has_insolvency_history"),
    col("jurisdiction"),
    col("last_full_members_list_date"),
    # flatten accounts
    col("accounts.next_accounts.overdue").alias("next_accounts_overdue"),
    col("accounts.overdue").alias("accounts_overdue"),
    col("confirmation_statement.overdue").alias("confirmation_statement_overdue"),
    col("registered_office_address.address_line_1").alias("address_line_1"),
    col("registered_office_address.country").alias("country"),
    col("registered_office_address.locality").alias("locality"),
    col("registered_office_address.postal_code").alias("postal_code"),
    col("last_updated_ts")
)

def scd_merge_table(spark, source_table, target_table, business_key):
    

    if not spark.catalog.tableExists(target_table):

        print("first load : creating silver table")

        source_table.write.format("delta").mode("overwrite").saveAsTable(target_table)

        print("table created")
    else:
        print("incremental load : performing scd 1 merged")

        merge_condition = " AND " .join([f"target.{col}=source.{col}" for col in business_key])
        
        d_table = DeltaTable.forName(spark, target_table)
        d_table.alias("target")\
        .merge( source_table.alias("source"),  merge_condition)\
            .whenMatchedUpdateAll()\
            .whenNotMatchedInsertAll().execute()

target_table = ("investment_intelligence_platform.silver.companies")
source_table = df
business_key = ["company_number"]
scd_merge_table(spark,source_table,target_table, business_key)
print("done ")